<a href="https://colab.research.google.com/github/atul6999/generative-ai/blob/main/openai_batch_chat_completions_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenAI Batch API — Chat Completions in Google Colab

Clean notebook for asynchronous batch jobs using `/v1/chat/completions`.

What it does:
1. Builds a `.jsonl` batch file.
2. Uploads it with `purpose="batch"`.
3. Creates a Batch job against `/v1/chat/completions`.
4. Polls status safely.
5. Downloads output + error files.
6. Parses responses into a DataFrame/CSV.

Use this for offline jobs: dataset labeling, transcript cleanup, categorization, rewriting, extraction, evals.


## 0. Install + imports

In [ ]:
!pip -q install --upgrade openai pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.7/58.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 2.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.


In [ ]:
import os
import json
import time
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List, Optional

import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

try:
    from google.colab import userdata, drive
    IN_COLAB = True
except Exception:
    userdata = None
    drive = None
    IN_COLAB = False

print("IN_COLAB:", IN_COLAB)


## 1. Configure

Set `OPENAI_API_KEY` in Colab secrets first.

Colab:
- Left sidebar → 🔑 Secrets
- Add `OPENAI_API_KEY`
- Enable notebook access


In [ ]:
# ===== CONFIG =====

PROJECT_DIR = Path("/content/openai_batch_chat")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# Optional Drive persistence.
USE_DRIVE = False
DRIVE_DIR = Path("/content/drive/MyDrive/openai_batch_chat")

# Chat Completions model. Keep a model that supports Batch API.
MODEL = "gpt-5.4-nano-2026-03-17"  # change if needed

# Batch only supports 24h completion window.
COMPLETION_WINDOW = "24h"

# Generation params for Chat Completions.
TEMPERATURE = 0.2
MAX_COMPLETION_TOKENS = 512

# Poll interval. Do not hammer the API.
POLL_SECONDS = 60

# Files.
INPUT_CSV = PROJECT_DIR / "input.csv"
BATCH_INPUT_JSONL = PROJECT_DIR / "batch_input.jsonl"
BATCH_META_JSON = PROJECT_DIR / "batch_meta.json"
BATCH_OUTPUT_JSONL = PROJECT_DIR / "batch_output.jsonl"
BATCH_ERROR_JSONL = PROJECT_DIR / "batch_error.jsonl"
RESULTS_CSV = PROJECT_DIR / "results.csv"
RESULTS_JSONL = PROJECT_DIR / "results.jsonl"

if USE_DRIVE:
    drive.mount("/content/drive")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)

api_key = os.environ.get("OPENAI_API_KEY")

if not api_key and IN_COLAB:
    try:
        api_key = userdata.get("OPENAI_API_KEY")
    except Exception:
        api_key = None

if not api_key:
    raise RuntimeError("Missing OPENAI_API_KEY. Add it to Colab secrets or os.environ.")

client = OpenAI(api_key=api_key)

print("Project dir:", PROJECT_DIR)
print("Model:", MODEL)


## 2. Create or load input data

Expected input format:

| id | input_text |
|---|---|
| row_001 | text to process |

Replace this cell with your dataset load if needed.


In [ ]:
# Demo data. Replace with your own CSV load if needed.
demo_rows = [
    {"id": "row_001", "input_text": "The product onboarding feels confusing. I could not find where to start."},
    {"id": "row_002", "input_text": "Pricing is fair, but the checkout page failed twice."},
    {"id": "row_003", "input_text": "The course content is excellent. I want more advanced modules."},
]

df = pd.DataFrame(demo_rows)
df.to_csv(INPUT_CSV, index=False)

# To use your own file:
# df = pd.read_csv("/content/your_file.csv")
# assert {"id", "input_text"}.issubset(df.columns)

df.head()


## 3. Define prompt builder

Edit this for your task.

This example returns strict JSON for each row.


In [ ]:
SYSTEM_PROMPT = """You are a precise data labeling assistant.
Return only valid JSON. No markdown.
"""

def build_user_prompt(input_text: str) -> str:
    return f"""
Analyze the customer feedback below.

Return JSON with:
- sentiment: one of positive, neutral, negative, mixed
- main_issue: short phrase
- urgency: one of low, medium, high
- summary: one sentence

Feedback:
{input_text}
""".strip()

def make_chat_completion_body(input_text: str) -> Dict[str, Any]:
    return {
        "model": MODEL,
        "temperature": TEMPERATURE,
        "max_completion_tokens": MAX_COMPLETION_TOKENS,
        "response_format": {"type": "json_object"},
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(input_text)},
        ],
    }

body_preview = make_chat_completion_body(df.iloc[0]["input_text"])
json.dumps(body_preview, indent=2)[:2000]


## 4. Build Batch JSONL

Each line must include:
- `custom_id`
- `method`
- `url`
- `body`

Important: for Batch, `url` is exactly `/v1/chat/completions`.


In [ ]:
def safe_custom_id(x: Any, fallback_index: int) -> str:
    raw = str(x).strip() if pd.notna(x) else ""
    if not raw:
        raw = f"row_{fallback_index:06d}"
    # Keep IDs simple and stable.
    return raw.replace(" ", "_")[:512]

def write_batch_jsonl(df: pd.DataFrame, path: Path) -> None:
    required = {"id", "input_text"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    seen = set()
    with path.open("w", encoding="utf-8") as f:
        for i, row in df.reset_index(drop=True).iterrows():
            custom_id = safe_custom_id(row["id"], i)
            if custom_id in seen:
                custom_id = f"{custom_id}__dup_{i}"
            seen.add(custom_id)

            task = {
                "custom_id": custom_id,
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": make_chat_completion_body(str(row["input_text"])),
            }
            f.write(json.dumps(task, ensure_ascii=False) + "\n")

write_batch_jsonl(df, BATCH_INPUT_JSONL)

print("Wrote:", BATCH_INPUT_JSONL)
print("Size MB:", round(BATCH_INPUT_JSONL.stat().st_size / 1024 / 1024, 3))

# Preview first 2 lines.
with BATCH_INPUT_JSONL.open("r", encoding="utf-8") as f:
    for _, line in zip(range(2), f):
        print(json.dumps(json.loads(line), indent=2, ensure_ascii=False)[:2000])
        print("-" * 80)


Wrote: /content/openai_batch_chat/batch_input.jsonl
Size MB: 0.002
{
  "custom_id": "row_001",
  "method": "POST",
  "url": "/v1/chat/completions",
  "body": {
    "model": "gpt-5.4-nano-2026-03-17",
    "temperature": 0.2,
    "max_completion_tokens": 512,
    "response_format": {
      "type": "json_object"
    },
    "messages": [
      {
        "role": "system",
        "content": "You are a precise data labeling assistant.\nReturn only valid JSON. No markdown.\n"
      },
      {
        "role": "user",
        "content": "Analyze the customer feedback below.\n\nReturn JSON with:\n- sentiment: one of positive, neutral, negative, mixed\n- main_issue: short phrase\n- urgency: one of low, medium, high\n- summary: one sentence\n\nFeedback:\nThe product onboarding feels confusing. I could not find where to start."
      }
    ]
  }
}
--------------------------------------------------------------------------------
{
  "custom_id": "row_002",
  "method": "POST",
  "url": "/v1/chat/compl

## 5. Validate JSONL locally

This catches dumb issues before upload.


In [ ]:
def validate_batch_jsonl(path: Path) -> Dict[str, Any]:
    ids = set()
    n = 0
    errors = []

    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            try:
                obj = json.loads(line)
            except Exception as e:
                errors.append(f"Line {line_no}: invalid JSON: {e}")
                continue

            n += 1
            for key in ["custom_id", "method", "url", "body"]:
                if key not in obj:
                    errors.append(f"Line {line_no}: missing {key}")

            if obj.get("method") != "POST":
                errors.append(f"Line {line_no}: method must be POST")

            if obj.get("url") != "/v1/chat/completions":
                errors.append(f"Line {line_no}: url must be /v1/chat/completions")

            custom_id = obj.get("custom_id")
            if custom_id in ids:
                errors.append(f"Line {line_no}: duplicate custom_id {custom_id}")
            ids.add(custom_id)

            body = obj.get("body", {})
            if body.get("model") != MODEL:
                errors.append(f"Line {line_no}: model mismatch")
            if "messages" not in body:
                errors.append(f"Line {line_no}: missing body.messages")

    return {"num_lines": n, "num_errors": len(errors), "errors": errors[:20]}

validation = validate_batch_jsonl(BATCH_INPUT_JSONL)
validation


{'num_lines': 3, 'num_errors': 0, 'errors': []}

## 6. Upload batch file

In [ ]:
if validation["num_errors"] > 0:
    raise RuntimeError(validation)

batch_input_file = client.files.create(
    file=BATCH_INPUT_JSONL.open("rb"),
    purpose="batch",
)

print("Uploaded file id:", batch_input_file.id)
batch_input_file


Uploaded file id: file-G9e3qcSAUzAwWb5xFcyUHM


FileObject(id='file-G9e3qcSAUzAwWb5xFcyUHM', bytes=1974, created_at=1778247753, filename='batch_input.jsonl', object='file', purpose='batch', status='processed', expires_at=1780839753, status_details=None)

## 7. Create Batch job

In [ ]:
batch = client.batches.create(
    input_file_id=batch_input_file.id,
    endpoint="/v1/chat/completions",
    completion_window=COMPLETION_WINDOW,
    metadata={
        "description": "Colab Chat Completions batch job",
        "created_from": "openai_batch_chat_completions_colab",
    },
)

batch_meta = batch.model_dump() if hasattr(batch, "model_dump") else dict(batch)
BATCH_META_JSON.write_text(json.dumps(batch_meta, indent=2), encoding="utf-8")

print("Batch id:", batch.id)
print("Status:", batch.status)
print("Saved:", BATCH_META_JSON)
batch


Batch id: batch_69fde84989348190b75d6a27ddbb0168
Status: validating
Saved: /content/openai_batch_chat/batch_meta.json


Batch(id='batch_69fde84989348190b75d6a27ddbb0168', completion_window='24h', created_at=1778247753, endpoint='/v1/chat/completions', input_file_id='file-G9e3qcSAUzAwWb5xFcyUHM', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1778334153, failed_at=None, finalizing_at=None, in_progress_at=None, metadata={'description': 'Colab Chat Completions batch job', 'created_from': 'openai_batch_chat_completions_colab'}, model=None, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0), usage=BatchUsage(input_tokens=0, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=0, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=0))

## 8. Poll status

You can stop Colab and resume later if you saved `batch.id`.

Terminal states:
- `completed`
- `failed`
- `expired`
- `cancelled`


In [ ]:
# Resume manually if needed:
# BATCH_ID = "batch_xxx"

BATCH_ID = batch.id

TERMINAL_STATUSES = {"completed", "failed", "expired", "cancelled"}

def print_batch_status(b):
    request_counts = getattr(b, "request_counts", None)
    print(
        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "| status:", b.status,
        "| output_file_id:", getattr(b, "output_file_id", None),
        "| error_file_id:", getattr(b, "error_file_id", None),
        "| request_counts:", request_counts,
    )

while True:
    batch = client.batches.retrieve(BATCH_ID)
    print_batch_status(batch)

    batch_meta = batch.model_dump() if hasattr(batch, "model_dump") else dict(batch)
    BATCH_META_JSON.write_text(json.dumps(batch_meta, indent=2), encoding="utf-8")

    if batch.status in TERMINAL_STATUSES:
        break

    time.sleep(POLL_SECONDS)

batch


2026-05-08 13:42:34 | status: validating | output_file_id: None | error_file_id: None | request_counts: BatchRequestCounts(completed=0, failed=0, total=0)
2026-05-08 13:43:34 | status: validating | output_file_id: None | error_file_id: None | request_counts: BatchRequestCounts(completed=0, failed=0, total=0)
2026-05-08 13:44:34 | status: in_progress | output_file_id: None | error_file_id: None | request_counts: BatchRequestCounts(completed=0, failed=0, total=3)
2026-05-08 13:45:34 | status: completed | output_file_id: file-L8KLYyu6ka9RyeBFQ39xz1 | error_file_id: None | request_counts: BatchRequestCounts(completed=3, failed=0, total=3)


Batch(id='batch_69fde84989348190b75d6a27ddbb0168', completion_window='24h', created_at=1778247753, endpoint='/v1/chat/completions', input_file_id='file-G9e3qcSAUzAwWb5xFcyUHM', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1778247901, error_file_id=None, errors=None, expired_at=None, expires_at=1778334153, failed_at=None, finalizing_at=1778247900, in_progress_at=1778247815, metadata={'description': 'Colab Chat Completions batch job', 'created_from': 'openai_batch_chat_completions_colab'}, model='gpt-5.4-nano-2026-03-17', output_file_id='file-L8KLYyu6ka9RyeBFQ39xz1', request_counts=BatchRequestCounts(completed=3, failed=0, total=3), usage=BatchUsage(input_tokens=262, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=144, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=406))

## 9. Download output + error files

Output line order is not guaranteed. Always join by `custom_id`.


In [ ]:
def download_file_content(file_id: str, out_path: Path) -> None:
    content = client.files.content(file_id)
    # Current SDK returns an object with write_to_file; fallback to bytes/text.
    if hasattr(content, "write_to_file"):
        content.write_to_file(str(out_path))
    else:
        data = getattr(content, "content", None)
        if data is None:
            data = content.read()
        mode = "wb" if isinstance(data, (bytes, bytearray)) else "w"
        with out_path.open(mode) as f:
            f.write(data)

batch = client.batches.retrieve(BATCH_ID)

if batch.output_file_id:
    download_file_content(batch.output_file_id, BATCH_OUTPUT_JSONL)
    print("Downloaded output:", BATCH_OUTPUT_JSONL, "size:", BATCH_OUTPUT_JSONL.stat().st_size)

if batch.error_file_id:
    download_file_content(batch.error_file_id, BATCH_ERROR_JSONL)
    print("Downloaded errors:", BATCH_ERROR_JSONL, "size:", BATCH_ERROR_JSONL.stat().st_size)

if not batch.output_file_id and not batch.error_file_id:
    print("No output/error file available. Batch status:", batch.status)


Downloaded output: /content/openai_batch_chat/batch_output.jsonl size: 3042


## 10. Parse results

This extracts:
- `custom_id`
- HTTP status
- assistant text
- parsed JSON if possible
- usage tokens
- raw response/error


In [ ]:
def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        return []
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def extract_chat_result(obj: Dict[str, Any]) -> Dict[str, Any]:
    custom_id = obj.get("custom_id")
    error = obj.get("error")
    response = obj.get("response") or {}
    status_code = response.get("status_code")
    body = response.get("body") or {}

    content = None
    parsed = None
    usage = body.get("usage")

    try:
        content = body["choices"][0]["message"]["content"]
    except Exception:
        content = None

    if content:
        try:
            parsed = json.loads(content)
        except Exception:
            parsed = None

    row = {
        "custom_id": custom_id,
        "status_code": status_code,
        "content": content,
        "parsed_json": json.dumps(parsed, ensure_ascii=False) if parsed is not None else None,
        "prompt_tokens": usage.get("prompt_tokens") if isinstance(usage, dict) else None,
        "completion_tokens": usage.get("completion_tokens") if isinstance(usage, dict) else None,
        "total_tokens": usage.get("total_tokens") if isinstance(usage, dict) else None,
        "error": json.dumps(error, ensure_ascii=False) if error else None,
        "raw": json.dumps(obj, ensure_ascii=False),
    }

    if isinstance(parsed, dict):
        for k, v in parsed.items():
            row[f"json_{k}"] = v

    return row

output_rows = read_jsonl(BATCH_OUTPUT_JSONL)
error_rows = read_jsonl(BATCH_ERROR_JSONL)

parsed_rows = [extract_chat_result(x) for x in output_rows]

# Include error rows too.
for e in error_rows:
    parsed_rows.append({
        "custom_id": e.get("custom_id"),
        "status_code": None,
        "content": None,
        "parsed_json": None,
        "prompt_tokens": None,
        "completion_tokens": None,
        "total_tokens": None,
        "error": json.dumps(e.get("error") or e, ensure_ascii=False),
        "raw": json.dumps(e, ensure_ascii=False),
    })

results_df = pd.DataFrame(parsed_rows)

# Join with original inputs.
final_df = df.copy()
final_df["custom_id"] = [
    safe_custom_id(row["id"], i)
    for i, row in df.reset_index(drop=True).iterrows()
]

# If duplicate IDs were rewritten, do a safer fallback based on result custom_id only.
final_df = final_df.merge(results_df, on="custom_id", how="left")

final_df.to_csv(RESULTS_CSV, index=False)
final_df.to_json(RESULTS_JSONL, orient="records", lines=True, force_ascii=False)

print("Output rows:", len(output_rows))
print("Error rows:", len(error_rows))
print("Saved CSV:", RESULTS_CSV)
print("Saved JSONL:", RESULTS_JSONL)

final_df.head(20)


Output rows: 3
Error rows: 0
Saved CSV: /content/openai_batch_chat/results.csv
Saved JSONL: /content/openai_batch_chat/results.jsonl


,id,input_text,custom_id,status_code,content,parsed_json,prompt_tokens,completion_tokens,total_tokens,error,raw,json_sentiment,json_main_issue,json_urgency,json_summary
0,row_001,The product onboarding feels confusing. I coul...,row_001,200,"{\n ""sentiment"": ""negative"",\n ""main_issue"":...","{""sentiment"": ""negative"", ""main_issue"": ""Confu...",89,56,145,None,"{""id"": ""batch_req_69fde8dcffdc8190a7d6721ad01e...",negative,Confusing product onboarding,high,The customer reports that onboarding is confus...
1,row_002,"Pricing is fair, but the checkout page failed ...",row_002,200,"{""sentiment"":""negative"",""main_issue"":""Checkout...","{""sentiment"": ""negative"", ""main_issue"": ""Check...",86,48,134,None,"{""id"": ""batch_req_69fde8dcf3988190b7d567d3c0c5...",negative,Checkout page failed twice,high,Customer says pricing is fair but the checkout...
2,row_003,The course content is excellent. I want more a...,row_003,200,"{""sentiment"":""positive"",""main_issue"":""Need mor...","{""sentiment"": ""positive"", ""main_issue"": ""Need ...",87,40,127,None,"{""id"": ""batch_req_69fde8dcf30c8190a072bbcc911b...",positive,Need more advanced modules,low,The customer praises the course content but re...


## 11. Persist to Google Drive

Run this after outputs look good.


In [ ]:
if USE_DRIVE:
    import shutil

    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    for p in [
        INPUT_CSV,
        BATCH_INPUT_JSONL,
        BATCH_META_JSON,
        BATCH_OUTPUT_JSONL,
        BATCH_ERROR_JSONL,
        RESULTS_CSV,
        RESULTS_JSONL,
    ]:
        if p.exists():
            shutil.copy2(p, DRIVE_DIR / p.name)
            print("Copied:", p.name)

    print("Drive dir:", DRIVE_DIR)
else:
    print("USE_DRIVE=False. Set it to True in config and rerun config cell if you want Drive persistence.")


## 12. Rerun only failed / missing rows

Use this after parsing if you need a retry batch.


In [ ]:
failed_or_missing = final_df[
    final_df["content"].isna() |
    final_df["error"].notna() |
    (final_df["status_code"].fillna(0) != 200)
].copy()

print("Failed/missing rows:", len(failed_or_missing))
failed_or_missing[["id", "input_text", "status_code", "error"]].head(20)

# To retry:
# retry_df = failed_or_missing[["id", "input_text"]].copy()
# BATCH_INPUT_JSONL = PROJECT_DIR / "batch_input_retry.jsonl"
# write_batch_jsonl(retry_df, BATCH_INPUT_JSONL)
# Then rerun upload -> create batch -> poll -> download -> parse.


## 13. Cancel a batch if needed

Only run if you really want to cancel the current batch.


In [ ]:
# client.batches.cancel(BATCH_ID)
